# 🏥 Medical Insurance Cost Prediction
### A Guided Regression Project for Students

> **Scenario**: You’re a data scientist at **HealthSure**, a health insurance startup. Your task is to **build a model that predicts annual insurance charges** for new customers.
>
> Why? To **automate quotes**, reduce manual underwriting, and improve customer experience.

---

## 🎯 What You’ll Learn
- How to handle **mixed data types** (numeric + categorical)
- Why we **transform skewed targets**
- How to **build and evaluate** a regression neural network
- How to **interpret real-world model performance**

👉 **Work through each step below. Run the provided code cells, then answer the questions in your own words.**

## 🔍 Step 1: Understand the Dataset

The **Medical Insurance Costs** dataset contains:
- **1,338 customers**
- **Features**: `age`, `sex`, `bmi`, `children`, `smoker`, `region`
- **Target**: `charges` (annual insurance cost in USD)

✅ **Run the cell below** to load and inspect the data.

> 💡 **Question**: What do you notice about the `charges` column? Are most values low or high?

In [ ]:
#@title Load and Preview Data
import pandas as pd

url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
df.head()

## 📊 Step 2: Explore the Target Variable

Insurance charges are **highly skewed**: most people pay modest amounts, but a few pay very high costs (e.g., due to chronic illness).

✅ **Run the cell below** to see the distribution.

> 💡 **Question**: Why might a **log-transform** help here? (Hint: Think about model assumptions and error symmetry.)

In [ ]:
#@title Plot Charges Distribution
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(df['charges'], bins=50, color='skyblue')
plt.title('Charges (Raw)')
plt.xlabel('Annual Charges (USD)')

plt.subplot(1, 2, 2)
plt.hist(np.log(df['charges']), bins=50, color='salmon')
plt.title('Charges (Log-Transformed)')
plt.xlabel('log(Charges)')

plt.tight_layout()
plt.show()

## 🧹 Step 3: Prepare the Data

We need to:
1. **Log-transform the target** → `y = log(charges)`
2. **Encode categorical features** (`sex`, `smoker`, `region`) → integers
3. **Split** into train/val/test
4. **Scale** features

✅ **Run the cell below** to apply these steps.

> 💡 **Question**: Why do we **encode categories as integers** instead of one-hot? (Hint: Consider dataset size and neural network flexibility.)

In [ ]:
#@title Preprocess Data (Hidden Details)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Log-transform target
y = np.log(df['charges'])
X = df.drop('charges', axis=1)

# Encode categoricals
X_processed = X.copy()
label_encoders = {}
for col in ['sex', 'smoker', 'region']:
    le = LabelEncoder()
    X_processed[col] = le.fit_transform(X[col])
    label_encoders[col] = le

# Split
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_tsplit(X_train, y_train, test_size=0.25, random_state=42)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("✅ Preprocessing complete!")

## 🧠 Step 4: Build the Neural Network

We’ll use a simple architecture:
- **Input**: 6 features
- **Hidden layers**: 64 → 32 neurons (ReLU + Dropout)
- **Output**: 1 neuron (linear)

✅ **Run the cell below** to define and compile the model.

> 💡 **Question**: Why do we use **Dropout**? What problem does it solve?

In [ ]:
#@title Build Model
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_model():
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(6,)),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

model = build_model()
model.summary()

## 📈 Step 5: Train the Model

We’ll train for up to 100 epochs, but stop early if validation loss stops improving.

✅ **Run the cell below** to train.

> 💡 **Question**: Look at the **validation MAE**. Is it decreasing? If it starts increasing, what does that mean?

In [ ]:
#@title Train Model
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

## 📊 Step 6: Evaluate Performance

We’ll evaluate on the **original USD scale** (not log-scale) for real-world interpretability.

Key metrics:
- **MAE**: Average error in dollars
- **MAPE**: Average % error
- **R²**: % of variance explained

✅ **Run the cell below** to see results.

> 💡 **Question**: If MAPE is 12%, what does that mean for a $10,000 policy? Is this acceptable for a business?

In [ ]:
#@title Evaluate Model
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

y_pred_log = model.predict(X_test_scaled, verbose=0).flatten()
y_test_usd = np.exp(y_test)
y_pred_usd = np.exp(y_pred_log)

mae = mean_absolute_error(y_test_usd, y_pred_usd)
mape = mean_absolute_percentage_error(y_test_usd, y_pred_usd) * 100
r2 = r2_score(y_test_usd, y_pred_usd)

print(f"🎯 Performance (USD scale):")
print(f"MAE  : ${mae:,.2f}")
print(f"MAPE : {mape:.2f}%")
print(f"R²   : {r2:.4f}")

## 🎛️ Step 7: Test Your Model

Use the interactive tool below to predict charges for a **custom customer profile**.

✅ **Adjust the sliders/dropdowns** and see real-time predictions.

> 💡 **Experiment**: How much does being a **smoker** increase the predicted cost? What about **age** or **BMI**?

In [ ]:
#@title Interactive Predictor
import ipywidgets as widgets
from IPython.display import display, clear_output
from contextlib import redirect_stdout
import os

age_slider = widgets.IntSlider(value=30, min=18, max=65, description='Age:')
sex_dropdown = widgets.Dropdown(options=['male', 'female'], value='female', description='Sex:')
bmi_slider = widgets.FloatSlider(value=25.0, min=15.0, max=50.0, step=0.1, description='BMI:')
children_slider = widgets.IntSlider(value=0, min=0, max=5, description='Children:')
smoker_dropdown = widgets.Dropdown(options=['no', 'yes'], value='no', description='Smoker:')
region_dropdown = widgets.Dropdown(options=['southwest', 'southeast', 'northwest', 'northeast'], value='southeast', description='Region:')

output = widgets.Output()

def predict_charges(change):
    with output:
        clear_output(wait=True)
        try:
            sex_enc = label_encoders['sex'].transform([sex_dropdown.value])[0]
            smoker_enc = label_encoders['smoker'].transform([smoker_dropdown.value])[0]
            region_enc = label_encoders['region'].transform([region_dropdown.value])[0]
            features = [[age_slider.value, sex_enc, bmi_slider.value, children_slider.value, smoker_enc, region_enc]]
            features_scaled = scaler.transform(features)
            with redirect_stdout(open(os.devnull, 'w')):
                pred_log = model.predict(features_scaled, verbose=0)[0][0]
            pred_usd = np.exp(pred_log)
            print(f"💲 Predicted Annual Charges: ${pred_usd:,.2f}")
        except Exception as e:
            print(f"⚠️ Error: {str(e)}")

for widget in [age_slider, sex_dropdown, bmi_slider, children_slider, smoker_dropdown, region_dropdown]:
    widget.observe(predict_charges, names='value')

display(widgets.VBox([
    widgets.HTML("<h3>✨ Test Your Insurance Quote</h3>"),
    age_slider, sex_dropdown, bmi_slider, children_slider, smoker_dropdown, region_dropdown, output
]))
predict_charges(None)

## 🎓 Reflection Questions

Answer these in your notebook or discussion:

1. **Data**: Why was log-transforming the target important?
2. **Modeling**: How did categorical encoding impact the model?
3. **Evaluation**: Is MAPE more useful than MAE for this business problem? Why?
4. **Ethics**: Could this model unfairly discriminate? How would you check?

✅ **Well done!** You’ve built a real-world regression model from start to finish.